In [ ]:
# Russian Classical Baselines

# This notebook establishes the classical machine-learning baselines for CWE classification in the Russian domain.
# Model and text-representation selection is performed using the fixed Russian validation set
# before final evaluation on the held-out Russian test set.

In [1]:
import numpy as np
import pandas as pd

from sklearn.dummy import (
    DummyClassifier
)

from src.config import (
    TEXT_COL,
    LABEL_COL,
    ID_COL,
    CWE_LABELS,
    RESULTS_MAIN_DIR,
    RESULTS_PER_CLASS_DIR,
    RESULTS_PREDICTIONS_DIR,
)

from src.data import (
    load_preprocessed_data,
    select_split,
    validate_label_space,
)

from src.models import (
    fit_predict_classical,
)

from src.evaluation import (
    evaluate_predictions,
)


ru_df, _ = load_preprocessed_data()

ru_train = select_split(
    ru_df,
    "train"
)

ru_val = select_split(
    ru_df,
    "val"
)

ru_test = select_split(
    ru_df,
    "test"
)


validate_label_space(
    ru_train
)

validate_label_space(
    ru_val
)

validate_label_space(
    ru_test
)


print(
    len(ru_train),
    len(ru_val),
    len(ru_test)
)

10114 2166 2171


In [ ]:
## Majority-Class Baseline

# A most-frequent-class classifier is evaluated on the Russian validation set to provide
# a simple reference baseline for the 30-class classification task.

In [2]:
dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(
    np.zeros(
        (len(ru_train), 1)
    ),
    ru_train[LABEL_COL]
)

majority_val_pred = (
    dummy.predict(
        np.zeros(
            (len(ru_val), 1)
        )
    )
)

majority_val_metrics, _ = (
    evaluate_predictions(
        ru_val[LABEL_COL],
        majority_val_pred
    )
)

majority_val_metrics

{'macro_f1': 0.008015163823449769,
 'weighted_f1': 0.03285995140915695,
 'accuracy': 0.13665743305632502}

In [ ]:
## Compare Classical Model Configurations

# Logistic Regression and Linear SVM are evaluated with both word-level and character-level TF-IDF representations.
# Macro-F1 is used as the primary model-selection metric.

In [3]:
validation_results = []
validation_per_class = {}


experiments = [
    (
        "Logistic Regression",
        "word"
    ),
    (
        "Logistic Regression",
        "char"
    ),
    (
        "Linear SVM",
        "word"
    ),
    (
        "Linear SVM",
        "char"
    ),
]


for model_name, representation in experiments:

    prediction, n_features, runtime = (
        fit_predict_classical(
            train_df=ru_train,
            test_df=ru_val,
            model_name=model_name,
            representation=representation,
            text_col=TEXT_COL,
            label_col=LABEL_COL,
        )
    )

    metrics, per_class = (
        evaluate_predictions(
            ru_val[LABEL_COL],
            prediction
        )
    )

    validation_results.append({
        "model":
            model_name,

        "representation":
            representation,

        **metrics,

        "n_features":
            n_features,

        "runtime_seconds":
            runtime,
    })

    validation_per_class[
        f"{model_name}_{representation}"
    ] = per_class


validation_results_df = (
    pd.DataFrame(
        validation_results
    )
    .sort_values(
        "macro_f1",
        ascending=False
    )
)


validation_results_df

,model,representation,macro_f1,weighted_f1,accuracy,n_features,runtime_seconds
3,Linear SVM,char,0.858217,0.895039,0.895199,69854,15.738722
2,Linear SVM,word,0.852137,0.893783,0.893352,12945,1.971359
0,Logistic Regression,word,0.846592,0.890342,0.887812,12945,3.309214
1,Logistic Regression,char,0.832234,0.874545,0.870268,69854,20.310726


In [ ]:
## Select the Best Text Representation

# The strongest validation representation is selected independently for
# Logistic Regression and Linear SVM according to Macro-F1.

In [4]:
BEST_LR_REPRESENTATION = (
    validation_results_df.loc[
        validation_results_df[
            "model"
        ] == "Logistic Regression"
    ]
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .iloc[0][
        "representation"
    ]
)


BEST_SVM_REPRESENTATION = (
    validation_results_df.loc[
        validation_results_df[
            "model"
        ] == "Linear SVM"
    ]
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .iloc[0][
        "representation"
    ]
)


print(
    "LR:",
    BEST_LR_REPRESENTATION
)

print(
    "SVM:",
    BEST_SVM_REPRESENTATION
)

LR: word
SVM: char


In [ ]:
## Final Russian-Domain Evaluation

# Training and validation data are combined into the Russian development set.
# The selected classical configurations are retrained on this full development
# set and evaluated once on the fixed Russian test set.

In [5]:
ru_dev = pd.concat(
    [
        ru_train,
        ru_val
    ],
    ignore_index=True
)


final_results = []
final_predictions = {}


for model_name, representation in [
    (
        "Logistic Regression",
        BEST_LR_REPRESENTATION
    ),
    (
        "Linear SVM",
        BEST_SVM_REPRESENTATION
    ),
]:

    prediction, n_features, runtime = (
        fit_predict_classical(
            ru_dev,
            ru_test,
            model_name,
            representation,
            TEXT_COL,
            LABEL_COL,
        )
    )

    metrics, per_class = (
        evaluate_predictions(
            ru_test[LABEL_COL],
            prediction
        )
    )

    final_results.append({
        "model":
            model_name,

        "representation":
            representation,

        **metrics,

        "n_features":
            n_features,

        "runtime_seconds":
            runtime,
    })

    final_predictions[
        model_name
    ] = prediction

    per_class.to_csv(
        RESULTS_PER_CLASS_DIR
        / f"RU_test_{model_name.replace(' ', '_')}.csv",
        index=False
    )


baseline_test_results = (
    pd.DataFrame(
        final_results
    )
)

baseline_test_results

,model,representation,macro_f1,weighted_f1,accuracy,n_features,runtime_seconds
0,Logistic Regression,word,0.866775,0.891821,0.889452,14943,4.506429
1,Linear SVM,char,0.884289,0.903213,0.902810,78459,18.713036


In [ ]:
## Save Baseline Results and Predictions

# Validation results, final Russian test metrics, per-class results,
# and record-level predictions are saved for subsequent cross-domain comparison and error analysis.

In [6]:
validation_results_df.to_csv(
    RESULTS_MAIN_DIR
    / "RU_baseline_validation_results.csv",
    index=False
)

baseline_test_results.to_csv(
    RESULTS_MAIN_DIR
    / "RU_baseline_test_results.csv",
    index=False
)


pd.DataFrame({
    ID_COL:
        ru_test[ID_COL].values,

    "true_cwe":
        ru_test[LABEL_COL].values,

    "logistic_regression_prediction":
        final_predictions[
            "Logistic Regression"
        ],

    "linear_svm_prediction":
        final_predictions[
            "Linear SVM"
        ],
}).to_csv(
    RESULTS_PREDICTIONS_DIR
    / "RU_baseline_test_predictions.csv",
    index=False
)